In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "two_particle_ho_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model and GFMC Parameters

This notebook runs fixed-node GFMC for two spinless fermions in a one-dimensional harmonic trap with configuration `R = (x1, x2)` and Hamiltonian
`H = -D (d^2/dx1^2 + d^2/dx2^2) + 0.5 * omega^2 * (x1^2 + x2^2)` with `D = 0.5`.

The exact fermionic ground state is proportional to `(x1 - x2) * exp(-0.5 * omega * (x1^2 + x2^2))`, so the nodal surface is `x1 = x2` and the exact energy is `E = 2 * omega`.

To make this a nontrivial fixed-node test, the guiding state below is intentionally imperfect:
`psi_T(R) = (x1 - x2) * exp(-0.25 * alpha_cm * (x1 + x2)^2 - 0.25 * alpha_rel * (x1 - x2)^2)`
with `alpha_cm != omega` and `alpha_rel != omega`.

Because the Gaussian factor is positive and symmetric, this trial state has the correct nodal surface and nonzero overlap with the exact fermionic ground state, but it is not the exact amplitude.

Parameters used below:
- Oscillator frequency `omega = 1.0`
- Imperfect trial widths `alpha_cm = 0.80 * omega`, `alpha_rel = 1.20 * omega`
- Time step `dt = 5.0e-3`
- Total steps `nsteps = 400`
- Equilibration steps `nequil = 60`
- Target population `targetN = 1500`
- Feedback strength `feedback = 0.2`
- Reconfiguration interval `reconfiguration_interval = 5`
- Branch-weight cap `branch_cap = 5.0`
- ET averaging window `energy_window = 50`

Trial / node structure:
- Guiding policy: `ImportanceGuiding(trial, H)`
- Node policy: `FixedNode()`
- `signpsi` returns `0` near `x1 = x2` to define the nodal surface


## Julia Construction

The next cell defines the explicit two-coordinate Hamiltonian, the imperfect antisymmetric trial state, the fixed-node policy, and plotting toggles.

The run cell executes GFMC. The final cell visualizes the energy history, the final walker cloud in `(x1, x2)`, the final coordinate marginals, and the relative-coordinate snapshot densities.


In [ ]:
omega = 1.0
# Deliberately imperfect: the exact fermionic amplitude would use alpha_cm = alpha_rel = omega.
alpha_cm = 0.80 * omega
alpha_rel = 1.20 * omega
node_tol = 1.0e-10

V(R) = begin
    x1, x2 = R
    0.5 * omega^2 * (x1^2 + x2^2)
end
H = Hamiltonian(2, 0.5, V)

function split_coords(R)
    x1, x2 = R
    S = x1 + x2
    Delta = x1 - x2
    return x1, x2, S, Delta
end

logpsi(R) = begin
    _, _, S, Delta = split_coords(R)
    abs_delta = abs(Delta)
    abs_delta < node_tol ? -Inf : log(abs_delta) - 0.25 * alpha_cm * S^2 - 0.25 * alpha_rel * Delta^2
end
gradlogpsi(R) = begin
    _, _, S, Delta = split_coords(R)
    abs_delta = abs(Delta)
    s = sign(Delta)
    abs_delta < node_tol ? Float64[0.0, 0.0] : Float64[
        (s / abs_delta) - 0.5 * alpha_cm * S - 0.5 * alpha_rel * Delta,
        -(s / abs_delta) - 0.5 * alpha_cm * S + 0.5 * alpha_rel * Delta,
    ]
end
lapllogpsi(R) = begin
    _, _, _, Delta = split_coords(R)
    abs_delta = abs(Delta)
    abs_delta < node_tol ? -Inf : -2.0 / abs_delta^2 - alpha_cm - alpha_rel
end
signpsi(R; tol::Float64=node_tol) = begin
    Delta = R[1] - R[2]
    abs(Delta) < tol ? 0.0 : sign(Delta)
end
trial = TrialWF(logpsi, gradlogpsi, lapllogpsi, signpsi)
guiding = ImportanceGuiding(trial, H)

EXACT_ENERGY = 2.0 * omega

function safe_fermion_configuration(rng; sep_min::Float64=1.0e-2, scale::Float64=1.0)
    x1 = scale * randn(rng)
    x2 = scale * randn(rng)
    while abs(x1 - x2) < sep_min
        x1 = scale * randn(rng)
        x2 = scale * randn(rng)
    end
    return [x1, x2]
end

relative_coordinates(snapshot) = Float64[R[1] - R[2] for R in snapshot]

targetN = 1500
dt = 5.0e-3
nsteps = 400
nequil = 60
ET0 = EXACT_ENERGY
feedback = 0.2
reconfiguration_interval = 5
branch_cap = 5.0
energy_window = 50

params = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
RECONFIGURATION = SystematicReconfiguration()

rng_init = MersenneTwister(1234)
initial_positions = [safe_fermion_configuration(rng_init) for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 140
DENSITY_SMOOTHING = 9
COORD_RANGE = nothing
RELATIVE_RANGE = nothing

RUN_LABEL = "guided fixed node (imperfect trial)"
RUN_COLOR = :teal
PLOT_TITLE = "Two-fermion HO GFMC"
STATE_TITLE = "Two-fermion HO GFMC: explicit x1/x2 fixed-node state"

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20
WRITE_RUN_CSV = false
CSV_FILENAME = "two_fermion_ho_fixed_node_imperfect_trial_gfmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "two_fermion_ho_fixed_node_imperfect_trial_gfmc"


In [ ]:
sim = GFMCSim(
    H,
    params,
    initial_positions,
    MersenneTwister(42);
    guiding=guiding,
    nodepolicy=FixedNode(),
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println(@sprintf("exact fermionic ground-state energy = %.8f", EXACT_ENERGY))
println(@sprintf("energy bias = %.3e", mean_energy - EXACT_ENERGY))
println(@sprintf("trial widths: alpha_cm = %.3f, alpha_rel = %.3f", alpha_cm, alpha_rel))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end

SIMS = [sim]
SIM_LABELS = [RUN_LABEL]
SIM_COLORS = [RUN_COLOR]


In [ ]:
history_fig = nb_plot_gfmc_history(SIMS; labels=SIM_LABELS, colors=SIM_COLORS, title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

if COORD_RANGE === nothing
    coord_values = Float64[]
    for sim in SIMS
        append!(coord_values, nb_all_coordinates(sim; coord=1))
        append!(coord_values, nb_all_coordinates(sim; coord=2))
    end
    coord_lo, coord_hi = nb_padded_limits(coord_values; pad_frac=0.12)
else
    coord_lo, coord_hi = COORD_RANGE
end

base_sim = SIMS[1]
if RELATIVE_RANGE === nothing
    relative_values = Float64[]
    for snapshot in base_sim.walker_positions_history
        append!(relative_values, relative_coordinates(snapshot))
    end
    relative_lo, relative_hi = nb_padded_limits(relative_values; pad_frac=0.12)
else
    relative_lo, relative_hi = RELATIVE_RANGE
end

final_snapshot = nb_last_snapshot(base_sim)
x1 = Float64[R[1] for R in final_snapshot]
x2 = Float64[R[2] for R in final_snapshot]

scatter_panel = scatter(
    x1,
    x2;
    xlabel="x1",
    ylabel="x2",
    title="Final walker cloud",
    markersize=2.5,
    alpha=0.35,
    color=RUN_COLOR,
    label="walkers",
    xlims=(coord_lo, coord_hi),
    ylims=(coord_lo, coord_hi),
)

plot!(scatter_panel, [coord_lo, coord_hi], [coord_lo, coord_hi]; color=:gray55, linestyle=:dash, linewidth=1.6, label="x1 = x2 node")

marginal_panel = plot(
    xlabel="coordinate",
    ylabel="density",
    title="Final coordinate marginals",
    legend=:topright,
    xlims=(coord_lo, coord_hi),
)
for (coord_idx, label, color) in ((1, "x1", :navy), (2, "x2", :darkorange))
    centers, density = nb_density_curve_from_snapshot(
        final_snapshot;
        coord=coord_idx,
        nbins=NBINS,
        xmin=coord_lo,
        xmax=coord_hi,
        smoothing_window=DENSITY_SMOOTHING,
    )
    plot!(marginal_panel, centers, density; label=label, color=color, linewidth=2.3)
end

relative_panel = plot(
    xlabel="x1 - x2",
    ylabel="density",
    title="Relative-coordinate snapshots",
    legend=:topright,
    xlims=(relative_lo, relative_hi),
)
snapshot_count = min(length(SNAPSHOT_STEPS), length(base_sim.walker_positions_history))
available_steps = SNAPSHOT_STEPS[1:snapshot_count]
snapshot_colors = palette(:viridis, snapshot_count)
for (i, (snapshot, step_idx)) in enumerate(zip(base_sim.walker_positions_history[1:snapshot_count], available_steps))
    centers, density = nb_density_curve(
        relative_coordinates(snapshot);
        nbins=NBINS,
        xmin=relative_lo,
        xmax=relative_hi,
        smoothing_window=DENSITY_SMOOTHING,
    )
    plot!(relative_panel, centers, density; label="step $(step_idx)", color=snapshot_colors[i], linewidth=2.2, alpha=0.90)
end
vline!(relative_panel, [0.0]; color=:gray55, linestyle=:dash, linewidth=1.3, label="node")

state_fig = plot(scatter_panel, marginal_panel, relative_panel; layout=(1, 3), size=(1600, 480), plot_title=STATE_TITLE)
display(state_fig)
nb_save_figure(state_fig, PATHS.figures_dir, FIGURE_STEM, "state"; enabled=SAVE_FIGURES)
